# Active Learning — SLEAP Fine-tuning

Iteratively improves the SLEAP model by adding manually corrected inference videos to the training dataset and fine-tuning from the current best checkpoint.

**Workflow:**
1. Select the base model and the corrected videos to add
2. Verify all corrected label files exist
3. Merge selected videos into the TRAIN dataset → generate embedded `.pkg.slp`
4. Build fine-tune config (lower LR, start from existing weights)
5. Train locally (CPU) or prepare files for Colab (GPU)
6. Verify output and update the Inference Pipeline

**Prerequisite:** Run the Inference Pipeline (Phases 1–4) for each new video and manually correct predictions in SLEAP before running this notebook.  
**Environment:** Run in the `sleap` conda environment (`conda activate sleap`).

In [33]:
from pathlib import Path
import datetime

# ── FLAGS — edit before running ───────────────────────────────────────────────

# Videos to add to the training set.
# Each entry is a folder name under Inference_Pipeline/Predictions/.
# Must have been run through Phases 1–4 and corrected in SLEAP.
NEW_VIDEOS = [
    "CantonS_unamp_Fly1.2",
    "CantonS_unamp_Fly3.1",
]

# Model folder under Inference_Pipeline/Models/SLEAP_Models/ to fine-tune from.
BASE_MODEL_NAME = "drosophila_unet64_setB_260407_092817"

# ── Dataset files ─────────────────────────────────────────────────────────────
# Update these if the dataset is extended or re-annotated.
ORIG_TRAIN_SLP_NAME = "Drosophila_TRAIN_set_setB.slp"
ORIG_VAL_SLP_NAME   = "Drosophila_VAL_set_setB.slp"
ORIG_TEST_SLP_NAME  = "Drosophila_TEST_set_setB.slp"

# ── Fine-tune hyperparameters ─────────────────────────────────────────────────
MAX_EPOCHS = 1    # fewer than original 200; early stopping still active
BATCH_SIZE = 6     # reduce to 4 if GPU runs out of memory
# LR is derived automatically from the base model in Step 3 (base_lr / 10)

# ── Run mode ──────────────────────────────────────────────────────────────────
MODE = "colab"   # "local"  -> train here (CPU; slow but no upload needed)
                 # "colab"  -> prepare files + print upload instructions for GPU training


## Step 1 — Verify corrected labels

In [34]:
import sleap_io as sio

BASE_DIR       = Path().resolve()
DATASET_DIR    = BASE_DIR / "SLEAP_Dataset"
MODELS_DIR     = BASE_DIR / "Models" / "SLEAP_Models"
BASE_MODEL_DIR = MODELS_DIR / BASE_MODEL_NAME
ORIG_TRAIN_SLP = DATASET_DIR / ORIG_TRAIN_SLP_NAME
ORIG_VAL_SLP   = DATASET_DIR / ORIG_VAL_SLP_NAME
_val_stem    = ORIG_VAL_SLP_NAME[:-4] if ORIG_VAL_SLP_NAME.endswith(".slp") else ORIG_VAL_SLP_NAME
ORIG_VAL_PKG = DATASET_DIR / (_val_stem + ".pkg.slp")
ORIG_TEST_SLP  = DATASET_DIR / ORIG_TEST_SLP_NAME

# ── Auto-patch TRAIN .slp video paths to current folder location ─────────────
LOCAL_VIDEO_DIR = BASE_DIR / "Videos" / "PROCESSED_Colormap_Videos"
_train_labels = sio.load_slp(str(ORIG_TRAIN_SLP))
_patched = 0
for v in _train_labels.videos:
    expected = LOCAL_VIDEO_DIR / Path(v.filename).name
    if Path(v.filename) != expected:
        if not expected.exists():
            raise FileNotFoundError(
                f"Video not found in local Videos folder: {expected.name}\n"
                f"  Expected at: {expected}"
            )
        v.filename = str(expected)
        _patched += 1
if _patched:
    sio.save_slp(_train_labels, str(ORIG_TRAIN_SLP))
    print(f"Auto-patched {_patched} video path(s) in {ORIG_TRAIN_SLP.name}")

# ── Auto-patch TEST .slp video paths to current folder location ──────────────
if ORIG_TEST_SLP.exists():
    _test_labels = sio.load_slp(str(ORIG_TEST_SLP))
    _patched_test = 0
    for v in _test_labels.videos:
        expected = LOCAL_VIDEO_DIR / Path(v.filename).name
        if Path(v.filename) != expected:
            if not expected.exists():
                print(f"Warning: test video not found locally: {expected.name} -- skipping patch")
                break
            v.filename = str(expected)
            _patched_test += 1
    if _patched_test:
        sio.save_slp(_test_labels, str(ORIG_TEST_SLP))
        print(f"Auto-patched {_patched_test} video path(s) in {ORIG_TEST_SLP.name}")

# ── Check base model and dataset files ───────────────────────────────────────
for label, path in [
    ("Base model dir", BASE_MODEL_DIR),
    ("best.ckpt",      BASE_MODEL_DIR / "best.ckpt"),
    ("Original TRAIN", ORIG_TRAIN_SLP),
    ("Original VAL",   ORIG_VAL_SLP),
]:
    if not Path(path).exists():
        raise FileNotFoundError(f"{label} not found: {path}")
if not ORIG_TEST_SLP.exists():
    print(f"Warning: TEST labels not found: {ORIG_TEST_SLP.name} (update ORIG_TEST_SLP_NAME in FLAGS)")

# ── Auto-generate VAL .pkg.slp if not yet embedded ─────────────────
if not ORIG_VAL_PKG.exists():
    print(f"Embedding VAL frames → {ORIG_VAL_PKG.name} (one-time, ~1-2 min)...")
    _val_labels = sio.load_slp(str(ORIG_VAL_SLP))
    sio.save_slp(_val_labels, str(ORIG_VAL_PKG), embed=True)
    print(f"  Done: {ORIG_VAL_PKG.name}  ({ORIG_VAL_PKG.stat().st_size/1e6:.1f} MB)")
else:
    print(f"VAL pkg: {ORIG_VAL_PKG.name}  ({ORIG_VAL_PKG.stat().st_size/1e6:.1f} MB)")

# ── Locate corrected .slp for each video ─────────────────────────────────────
pred_files = {}
for vfn in NEW_VIDEOS:
    pred_dir = BASE_DIR / "Predictions" / vfn
    if not pred_dir.exists():
        raise FileNotFoundError(
            f"Predictions folder not found for '{vfn}':\n  {pred_dir}\n"
            "Run the Inference Pipeline (Phases 1-4) first."
        )
    slp_files = sorted(pred_dir.glob("*.slp"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not slp_files:
        raise FileNotFoundError(
            f"No .slp file found for '{vfn}' in:\n  {pred_dir}\n"
            "Complete Phase 4 (manual correction in SLEAP) first."
        )
    pred_files[vfn] = slp_files[0]
    if len(slp_files) > 1:
        print(f"[{vfn}] Multiple .slp files -- using most recent: {slp_files[0].name}")

# ── Summary ───────────────────────────────────────────────────────────────────
orig_labels = sio.load_slp(str(ORIG_TRAIN_SLP))
print(f"Base model       : {BASE_MODEL_NAME}")
print(f"Original TRAIN   : {ORIG_TRAIN_SLP.name}  ({len(orig_labels)} frames)")
print(f"Original VAL     : {ORIG_VAL_SLP.name}  →  pkg: {ORIG_VAL_PKG.name}")
print(f"Videos to add    : {len(NEW_VIDEOS)}")
for vfn, pf in pred_files.items():
    lbl = sio.load_slp(str(pf))
    print(f"  {vfn}: {pf.name}  ({len(lbl)} frames)")
print()
print("Proceed to Step 2 to merge and generate the dataset.")


[CantonS_unamp_Fly1.2] Multiple .slp files -- using most recent: CantonS_unamp_Fly1.2.predictions.slp
[CantonS_unamp_Fly3.1] Multiple .slp files -- using most recent: CantonS_unamp_Fly3.1.predictions.slp
Base model       : drosophila_unet64_setB_260407_092817
Original TRAIN   : Drosophila_TRAIN_set_setB.slp  (755 frames)
Original VAL     : Drosophila_VAL_set_setB.pkg.slp
Videos to add    : 2
  CantonS_unamp_Fly1.2: CantonS_unamp_Fly1.2.predictions.slp  (100 frames)
  CantonS_unamp_Fly3.1: CantonS_unamp_Fly3.1.predictions.slp  (121 frames)

Proceed to Step 2 to merge and generate the dataset.


## Step 2 — Merge datasets and generate embedded package
Adds all selected videos to the original TRAIN set and produces a `.pkg.slp` ready for training.

Two things are handled automatically:
- **PredictedInstance → Instance**: converts model predictions to user instances (blue in SLEAP) so they are treated as labeled training data
- **Empty frame filtering**: removes frames where all keypoints are invisible/NaN (cleared by Phase 4.4 trim) so the model never trains on empty frames

After merging, SLEAP GUI opens automatically so you can verify the labels. Close it when done and proceed to Step 3.

In [35]:
import numpy as np, h5py, json, re

def _to_instance(pred_inst):
    """Convert PredictedInstance (grey) to Instance (blue) for training."""
    return sio.Instance(
        skeleton=pred_inst.skeleton,
        points=pred_inst.points,
        track=pred_inst.track,
        from_predicted=None,
    )

def _has_visible_keypoints(lf):
    for inst in lf.instances:
        for row in inst.points.tolist():
            xy, visible = row[0], bool(row[1])
            if visible and not (np.isnan(float(xy[0])) or np.isnan(float(xy[1]))):
                return True
    return False

def _clean_labels(labels, source_name):
    clean_frames = []
    n_converted = 0
    n_filtered  = 0
    for lf in labels:
        new_instances = []
        for inst in lf.instances:
            if isinstance(inst, sio.PredictedInstance):
                new_instances.append(_to_instance(inst))
                n_converted += 1
            else:
                new_instances.append(inst)
        lf_clean = sio.LabeledFrame(
            video=lf.video,
            frame_idx=lf.frame_idx,
            instances=new_instances,
        )
        if _has_visible_keypoints(lf_clean):
            clean_frames.append(lf_clean)
        else:
            n_filtered += 1
    if n_converted:
        print(f"  [{source_name}] Converted {n_converted} PredictedInstance(s) -> Instance")
    if n_filtered:
        print(f"  [{source_name}] Filtered {n_filtered} empty/cleared frame(s)")
    return clean_frames

def _inject_suggestions(slp_path, merged_labels):
    """Write suggestions_json so every labeled frame shows as a blue tick in SLEAP GUI."""
    video_index = {v: i for i, v in enumerate(merged_labels.videos)}
    suggestions = []
    for lf in merged_labels:
        vid_idx = video_index[lf.video]
        suggestions.append(
            json.dumps({"video": str(vid_idx), "frame_idx": int(lf.frame_idx), "group": 0}).encode()
        )
    with h5py.File(str(slp_path), "a") as f:
        del f["suggestions_json"]
        f.create_dataset("suggestions_json", data=suggestions)
    return len(suggestions)

def _get_backbone_tag(model_dir):
    """Extract backbone name + filters from training_config.yaml for run naming."""
    cfg_path = Path(model_dir) / "training_config.yaml"
    try:
        import yaml
        with open(cfg_path) as f:
            cfg = yaml.safe_load(f)
        bb = cfg.get("model_config", {}).get("backbone_config", {})
        for name, val in bb.items():
            if val is not None:
                filters = val.get("filters", "")
                return f"{name}{filters}" if filters else name
    except Exception:
        pass
    # Fallback: parse from folder name
    m = re.search(r"(unet\d+|convnext|swint)", Path(model_dir).name)
    return m.group(1) if m else "model"

# -- Merge --------------------------------------------------------------------
all_frames = _clean_labels(orig_labels, "TRAIN")
n_orig = len(all_frames)

for vfn, pf in pred_files.items():
    new_lbl = sio.load_slp(str(pf))
    clean   = _clean_labels(new_lbl, vfn)
    all_frames += clean
    print(f"  + {vfn}: {len(clean)} usable frames  (raw: {len(new_lbl)})")

# -- Build run name: drosophila_<backbone>_ft_<Nframes>frames_<timestamp> -----
TIMESTAMP      = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
N_FRAMES       = len(all_frames)
_backbone_tag  = _get_backbone_tag(BASE_MODEL_DIR)
RUN_NAME       = f"drosophila_{_backbone_tag}_ft_{N_FRAMES}frames_{TIMESTAMP}"
NEW_MODEL_DIR  = MODELS_DIR / RUN_NAME
MERGED_SLP     = DATASET_DIR / f"Drosophila_TRAIN_set_setB_{N_FRAMES}frames.slp"
MERGED_PKG     = DATASET_DIR / f"Drosophila_TRAIN_set_setB_{N_FRAMES}frames.pkg.slp"

merged = sio.Labels(all_frames)
sio.save_slp(merged, str(MERGED_SLP))
print(f"\nMerged .slp saved : {MERGED_SLP.name}")
print(f"  Original TRAIN  : {n_orig} frames")
print(f"  New videos      : {N_FRAMES - n_orig} frames")
print(f"  Total           : {N_FRAMES} frames")

# -- Inject suggestions (blue ticks in SLEAP GUI timeline) --------------------
n_sug = _inject_suggestions(MERGED_SLP, merged)
print(f"Suggestions injected: {n_sug} (all frames blue in SLEAP timeline)")

# -- HDF5 sanity check --------------------------------------------------------
with h5py.File(str(MERGED_SLP), "r") as f:
    inst_types  = f["instances"]["instance_type"][:]
    n_sug_check = len(f["suggestions_json"][:])
n_predicted = int((inst_types == 1).sum())
if n_predicted:
    print(f"WARNING: {n_predicted} instance(s) still have instance_type=1 (grey)")
else:
    print(f"HDF5 check: all {len(inst_types)} instances are instance_type=0 (blue)")
print(f"HDF5 check: {n_sug_check} suggestion entries written")

print("\nEmbedding frames into .pkg.slp ...")
import gc; gc.collect()  # release any lingering HDF5 handles
if MERGED_PKG.exists():
    try:
        MERGED_PKG.unlink()
    except PermissionError:
        raise PermissionError(
            f"Cannot overwrite {MERGED_PKG.name} -- file is open (e.g. SLEAP GUI).\n"
            "Close the program using it and re-run this cell."
        )
sio.save_slp(merged, str(MERGED_PKG), embed=True)
_inject_suggestions(MERGED_PKG, merged)
size_mb = MERGED_PKG.stat().st_size / 1e6
print(f"Embedded package  : {MERGED_PKG.name}  ({size_mb:.1f} MB)")
print(f"RUN_NAME          : {RUN_NAME}")
print("Step 2 complete.")


  + CantonS_unamp_Fly1.2: 100 usable frames  (raw: 100)
  + CantonS_unamp_Fly3.1: 121 usable frames  (raw: 121)

Merged .slp saved : Drosophila_TRAIN_set_setB_976frames.slp
  Original TRAIN  : 755 frames
  New videos      : 221 frames
  Total           : 976 frames
Suggestions injected: 976 (all frames blue in SLEAP timeline)
HDF5 check: all 976 instances are instance_type=0 (blue)
HDF5 check: 976 suggestion entries written

Embedding frames into .pkg.slp ...


Embedding frames: 100%|██████████| 976/976 [00:03<00:00, 284.41it/s]


Embedded package  : Drosophila_TRAIN_set_setB_976frames.pkg.slp  (3.6 MB)
Run name          : drosophila_unet64_ft_976frames_260415_130816
Step 2 complete.


## Step 2.5 — Inspect merged dataset in SLEAP GUI (optional)
Run this cell to open the merged dataset in SLEAP. Verify that all frames show blue ticks on the timeline. Close SLEAP when done, then proceed to Step 3.

In [29]:
import subprocess, shutil

sleap_label = shutil.which("sleap-label")
if sleap_label:
    print(f"Opening {MERGED_SLP.name} in SLEAP GUI...")
    subprocess.Popen([sleap_label, str(MERGED_SLP)])
else:
    print("sleap-label not found in PATH. Open manually:")
    print(f'  sleap-label "{MERGED_SLP}"')


Opening Drosophila_TRAIN_set_setB_976frames.slp in SLEAP GUI...


## Step 3 — Build fine-tune config
Reads all parameters from the base model's `training_config.yaml` (backbone, sigma, augmentation, skeleton) and builds a new config with:
- Pretrained weights pointing to the base model's `best.ckpt`
- LR set to base model LR ÷ 10 (avoids catastrophic forgetting)
- Reduced max epochs with early stopping still active
- Output saved to a new folder — original model is never overwritten

In [36]:
import yaml, copy, json
try:
    import torch as _torch; _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False

CONFIG_PATH       = DATASET_DIR / 'single_instance_ft.yaml'
COLAB_CONFIG_PATH = DATASET_DIR / 'colab_config.json'

# Load base model config -- all parameters are inherited from here
with open(BASE_MODEL_DIR / 'training_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cfg = copy.deepcopy(cfg)

# Derive LR from base model (fine-tune at 1/10 to avoid catastrophic forgetting)
BASE_LR = cfg['trainer_config']['optimizer']['lr']
LR      = BASE_LR / 10

if MODE == 'local':
    TRAIN_PATH_CFG = str(MERGED_PKG).replace('\\', '/')
    VAL_PATH_CFG   = str(ORIG_VAL_PKG).replace('\\', '/')
    CKPT_PATH_CFG  = str(BASE_MODEL_DIR / 'best.ckpt').replace('\\', '/')
    CKPT_DIR_CFG   = str(MODELS_DIR).replace('\\', '/')
    ACCELERATOR    = 'gpu' if (_HAS_TORCH and _torch.cuda.is_available()) else 'cpu'
else:  # colab
    DRIVE_DIR      = '/content/drive/MyDrive/sleap_active_learning'
    TRAIN_PATH_CFG = f'{DRIVE_DIR}/{MERGED_PKG.name}'
    VAL_PATH_CFG   = f'{DRIVE_DIR}/{ORIG_VAL_PKG.name}'
    CKPT_PATH_CFG  = f'{DRIVE_DIR}/best_ft_base.ckpt'
    CKPT_DIR_CFG   = f'{DRIVE_DIR}/models'
    ACCELERATOR    = 'gpu'

# Data
cfg['data_config']['train_labels_path']   = [TRAIN_PATH_CFG]
cfg['data_config']['val_labels_path']     = [VAL_PATH_CFG]
# Test set -- local uses .slp, Colab uses embedded .pkg.slp
ORIG_TEST_PKG = ORIG_TEST_SLP.with_suffix('').with_suffix('.pkg.slp')
if MODE == 'local' and ORIG_TEST_SLP.exists():
    cfg['data_config']['test_file_path'] = [ORIG_TEST_SLP.as_posix()]
elif MODE == 'colab' and ORIG_TEST_PKG.exists():
    cfg['data_config']['test_file_path'] = [f'{DRIVE_DIR}/{ORIG_TEST_PKG.name}']
cfg['data_config']['user_instances_only'] = False
cfg['data_config']['cache_workers']       = 0  # h5py cannot be pickled across processes on Windows

# DataLoader workers -- 0 on Windows (h5py/multiprocessing), 4 on Colab
_n_workers = 0 if MODE == 'local' else 4
cfg['trainer_config']['train_data_loader']['num_workers'] = _n_workers
cfg['trainer_config']['val_data_loader']['num_workers']   = _n_workers

# Pretrained weights -- fine-tune from base model
cfg['model_config']['pretrained_backbone_weights'] = CKPT_PATH_CFG
cfg['model_config']['pretrained_head_weights']     = CKPT_PATH_CFG

# Fine-tune schedule
cfg['trainer_config']['max_epochs']                      = MAX_EPOCHS
cfg['trainer_config']['optimizer']['lr']                 = LR
cfg['trainer_config']['train_data_loader']['batch_size'] = BATCH_SIZE
cfg['trainer_config']['val_data_loader']['batch_size']   = BATCH_SIZE
cfg['trainer_config']['trainer_accelerator']             = ACCELERATOR

# Output -- new folder, never overwrites original
cfg['trainer_config']['run_name']      = RUN_NAME
cfg['trainer_config']['wandb']['name'] = RUN_NAME
cfg['trainer_config']['ckpt_dir']      = CKPT_DIR_CFG
cfg['name'] = RUN_NAME

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

# Colab config -- always generated so user can switch modes without re-running Steps 1-3
colab_cfg = {
    'run_name':        RUN_NAME,
    'merged_pkg_name': MERGED_PKG.name,
    'val_pkg_name':    ORIG_VAL_PKG.name,
    'test_pkg_name':   ORIG_TEST_PKG.name if ORIG_TEST_PKG.exists() else None,
    'base_ckpt_name':  'best_ft_base.ckpt',
    'yaml_name':       CONFIG_PATH.name,
    'n_frames':        N_FRAMES,
    'base_model_name': BASE_MODEL_NAME,
}
with open(COLAB_CONFIG_PATH, 'w') as f:
    json.dump(colab_cfg, f, indent=2)

backbone = next(k for k, v in cfg['model_config']['backbone_config'].items() if v is not None)
print(f'Config saved  : {CONFIG_PATH.name}')
print(f'Colab config  : {COLAB_CONFIG_PATH.name}')
print(f'  Backbone    : {backbone}  (inherited from base model)')
print(f'  LR          : {LR:.2e}  (base {BASE_LR:.2e} / 10)')
print(f'  Max epochs  : {MAX_EPOCHS}  (base: 200; early stopping still active)')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Accelerator : {ACCELERATOR}')
print(f'  num_workers : {_n_workers}  (0=Windows h5py, 4=Colab)')
print(f'  Train data  : {MERGED_PKG.name}  ({N_FRAMES} frames)')
print(f'  Val data    : {ORIG_VAL_PKG.name}')
if MODE == 'local' and ORIG_TEST_SLP.exists():
    print(f'  Test data   : {ORIG_TEST_SLP.name}')
elif MODE == 'colab' and ORIG_TEST_PKG.exists():
    print(f'  Test data   : {ORIG_TEST_PKG.name}  (embedded pkg)')
else:
    print(f'  Test data   : not found -- skipped')
print(f'  Pretrained  : {BASE_MODEL_NAME}/best.ckpt')
print(f'  Output      : {RUN_NAME}')
print('Step 3 complete. Proceed to Step 3.5 (hardware check) then Step 4.')

Config saved  : single_instance_ft.yaml
Colab config  : colab_config.json
  Backbone    : unet  (inherited from base model)
  LR          : 1.00e-05  (base 1.00e-04 / 10)
  Max epochs  : 1  (base: 200; early stopping still active)
  Batch size  : 6
  Accelerator : gpu
  num_workers : 0  (Windows h5py compatibility)
  Train data  : Drosophila_TRAIN_set_setB_976frames.pkg.slp  (976 frames)
  Val data    : Drosophila_VAL_set_setB.pkg.slp
  Test data   : Drosophila_TEST_set_setB.pkg.slp  (embedded pkg)
  Pretrained  : drosophila_unet64_setB_260407_092817/best.ckpt
  Output      : drosophila_unet64_ft_976frames_260415_130816
Step 3 complete. Proceed to Step 3.5 (hardware check) then Step 4.


## Step 3.5 — Hardware check
Detects available GPU/CPU and gives a training time estimate before you commit to running.

In [37]:
import platform, psutil
try:
    import torch as _torch; _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False

print("=" * 50)
print("Hardware report")
print("=" * 50)

cpu_name    = platform.processor() or platform.machine()
cpu_cores   = psutil.cpu_count(logical=False)
cpu_threads = psutil.cpu_count(logical=True)
ram_gb      = psutil.virtual_memory().total / 1e9
print(f"CPU   : {cpu_name}")
print(f"Cores : {cpu_cores} physical / {cpu_threads} logical")
print(f"RAM   : {ram_gb:.1f} GB")
print()

if _HAS_TORCH and _torch.cuda.is_available():
    for i in range(_torch.cuda.device_count()):
        props = _torch.cuda.get_device_properties(i)
        vram  = props.total_memory / 1e9
        print(f"GPU {i} : {props.name}  ({vram:.1f} GB VRAM)  OK")
    print()
    print("GPU detected — Step 3 set accelerator to 'gpu' automatically.")
elif not _HAS_TORCH:
    print("GPU   : torch not available in this environment")
else:
    print("GPU   : not available (CPU-only)")
    print()
    if MODE == "colab":
        print("Step 4 will prepare the upload zip for Colab GPU training.")

ACCELERATOR = "gpu" if (_HAS_TORCH and _torch.cuda.is_available()) else "cpu"
print()
print(f"Selected mode : {MODE.upper()}")
print(f"Accelerator   : {ACCELERATOR.upper()}")

Hardware report
CPU   : Intel64 Family 6 Model 142 Stepping 10, GenuineIntel
Cores : 4 physical / 8 logical
RAM   : 8.5 GB

GPU   : not available (CPU-only)

Step 4 will prepare the upload zip for Colab GPU training.

Selected mode : COLAB
Accelerator   : GPU


## Step 4 — Train
**Local mode:** runs training directly via subprocess. Slow on CPU. 
**Colab mode:** bundles all needed files into `colab_upload.zip` and opens Google Drive. Upload the zip, open `Active_Learning_Colab.ipynb` in Colab, and run all cells.

In [38]:
import subprocess, sys, zipfile, webbrowser, os

if MODE == 'local':
    import shutil as _shutil, subprocess as _subp2

    def _check_sleap_nn(py):
        """Return True if py can import sleap_nn."""
        try:
            r = _subp2.run([str(py), '-c', 'import sleap_nn'], capture_output=True, timeout=15)
            return r.returncode == 0
        except Exception:
            return False

    def _find_conda_envs():
        """Yield python.exe paths from all conda environments (base + named envs)."""
        for _base in (Path.home() / 'anaconda3', Path.home() / 'miniconda3',
                      Path('C:/ProgramData/anaconda3'), Path('C:/ProgramData/miniconda3')):
            # base conda env
            _py = _base / 'python.exe'
            if _py.exists():
                yield _py
            # named envs
            _envs_dir = _base / 'envs'
            if _envs_dir.exists():
                for _d in _envs_dir.iterdir():
                    _py = _d / 'python.exe'
                    if _py.exists():
                        yield _py

    _python = None
    # 1. sleap_nn importable (GUI running inside the sleap_nn env)
    try:
        import sleap_nn as _sleap_nn_mod
        _env_root = Path(_sleap_nn_mod.__file__).parents[3]
        _cand = _env_root / 'python.exe'
        if _cand.exists(): _python = _cand
    except ImportError:
        pass
    # 2. Scan all conda envs for one with sleap_nn installed
    if _python is None:
        print('Searching conda envs for sleap_nn...', flush=True)
        for _cand in _find_conda_envs():
            print(f'  Checking {_cand.parent.name}...', flush=True)
            if _check_sleap_nn(_cand):
                _python = _cand
                print(f'  Found sleap_nn in: {_cand}')
                break
    if _python is None:
        print()
        print('ERROR: sleap_nn not found in any conda environment.')
        print()
        print('To enable local training, install sleap-nn into an existing env:')
        print('  conda activate sleap_new')
        print('  pip install sleap-nn')
        print()
        print('Or switch to Colab mode (recommended) to train on GPU.')
        sys.exit(1)

    print(f'Starting fine-tuning (local {ACCELERATOR.upper()})...')
    print(f'Python : {_python}')
    print(f'Run    : {RUN_NAME}')
    print(f'Output : {NEW_MODEL_DIR}\n')

    # Prevent torch from loading CUDA DLLs when training on CPU
    _env = os.environ.copy()
    if ACCELERATOR == 'cpu':
        _env['CUDA_VISIBLE_DEVICES'] = '-1'

    # Stream output live so epoch progress is visible
    proc = subprocess.Popen(
        [str(_python), '-m', 'sleap_nn.cli', 'train', str(CONFIG_PATH)],
        cwd=str(BASE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=_env,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    if proc.returncode != 0:
        print(f'\nTraining failed (exit code {proc.returncode}).')
    else:
        print('\nTraining complete. Proceed to Step 5.')

elif MODE == 'colab':
    zip_path = DATASET_DIR / 'colab_upload.zip'
    _D = "sleap_active_learning"  # folder created inside zip
    files_to_zip = [
        (MERGED_PKG,                   f"{_D}/{MERGED_PKG.name}"),
        (ORIG_VAL_PKG,                 f"{_D}/{ORIG_VAL_PKG.name}"),
        (CONFIG_PATH,                  f"{_D}/{CONFIG_PATH.name}"),
        (COLAB_CONFIG_PATH,            f"{_D}/{COLAB_CONFIG_PATH.name}"),
        (BASE_MODEL_DIR / 'best.ckpt', f"{_D}/best_ft_base.ckpt"),
    ]
    if ORIG_TEST_PKG.exists():
        files_to_zip.append((ORIG_TEST_PKG, f"{_D}/{ORIG_TEST_PKG.name}"))
    # Base model metrics -- for comparison table on Colab
    _BM = f"{_D}/base_metrics"
    for _fn in ["metrics.train.0.npz", "metrics.val.0.npz", "metrics.test.0.npz",
                "labels_pr.train.0.slp", "labels_gt.train.0.slp",
                "labels_pr.val.0.slp",   "labels_gt.val.0.slp",
                "labels_pr.test.0.slp",  "labels_gt.test.0.slp",
                "training_log.csv"]:
        _fp = BASE_MODEL_DIR / _fn
        if _fp.exists():
            files_to_zip.append((_fp, f"{_BM}/{_fn}"))
    print('Bundling files for Colab upload...')
    with zipfile.ZipFile(str(zip_path), 'w', zipfile.ZIP_DEFLATED) as zf:
        for src, arcname in files_to_zip:
            zf.write(str(src), arcname)
            size_mb = Path(src).stat().st_size / 1e6
            print(f'  + {arcname}  ({size_mb:.1f} MB)')
    total_mb = zip_path.stat().st_size / 1e6
    print(f'\nZip ready : {zip_path.name}  ({total_mb:.1f} MB)')
    print(f'Location  : {zip_path}')
    sep = '=' * 62
    print(f'\n{sep}')
    print('Next steps')
    print(sep)
    print()
    print('1. Upload colab_upload.zip to the ROOT of your Google Drive (MyDrive/).')
    print('   Colab will unzip it automatically -- no manual unzipping needed.')
    print()
    print('2. Open Active_Learning_Colab.ipynb in Google Colab.')
    print('   Runtime > Change runtime type > T4 GPU')
    print()
    print('3. Run all cells -- unzip + training starts automatically.')
    print()
    print('4. After training, download the model folder from Drive:')
    print(f'     MyDrive/sleap_active_learning/models/{RUN_NAME}/')
    print(f'   Place it in: {MODELS_DIR}')
    print()
    print('5. Run Step 5 to verify and see training curves.')
    print()
    print('Opening Google Drive and Colab in browser...')
    webbrowser.open('https://drive.google.com/drive/my-drive')
    webbrowser.open('https://colab.research.google.com/')

Bundling files for Colab upload...
  + sleap_active_learning/Drosophila_TRAIN_set_setB_976frames.pkg.slp  (3.6 MB)
  + sleap_active_learning/Drosophila_VAL_set_setB.pkg.slp  (0.4 MB)
  + sleap_active_learning/single_instance_ft.yaml  (0.0 MB)
  + sleap_active_learning/colab_config.json  (0.0 MB)
  + sleap_active_learning/best_ft_base.ckpt  (187.9 MB)
  + sleap_active_learning/Drosophila_TEST_set_setB.pkg.slp  (0.4 MB)
  + sleap_active_learning/base_metrics/metrics.train.0.npz  (0.1 MB)
  + sleap_active_learning/base_metrics/metrics.val.0.npz  (0.0 MB)
  + sleap_active_learning/base_metrics/labels_pr.train.0.slp  (0.3 MB)
  + sleap_active_learning/base_metrics/labels_gt.train.0.slp  (0.2 MB)
  + sleap_active_learning/base_metrics/labels_pr.val.0.slp  (0.1 MB)
  + sleap_active_learning/base_metrics/labels_gt.val.0.slp  (0.1 MB)
  + sleap_active_learning/base_metrics/training_log.csv  (0.0 MB)

Zip ready : colab_upload.zip  (179.2 MB)
Location  : C:\Users\pepev\Desktop\103665_THESIS_DL_Mo

## Step 5 — Evaluate & compare results

Loads the fine-tuned model and base model, computes metrics for **train / val / test** splits, and displays styled comparison tables (VAL and TEST) with green/red highlights.

Skip to the **Quick Recovery** cell below if the kernel died after Step 4 completed.

In [ ]:
import pandas as pd, numpy as np, matplotlib, re
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
import sleap_io as sio

# ── CONFIG ────────────────────────────────────────────────────────────────────
TAU_PX    = 5.0   # correction threshold (px) — errors above this count as a miss
SAVE_FIGS = True

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

KP_NAMES = ["head","thorax","abdomen",
            "forelegR","forelegL","midlegR","midlegL","hindlegR","hindlegL"]
N_KP  = len(KP_NAMES)
SHORT = [{"forelegR":"fR","forelegL":"fL","midlegR":"mR",
          "midlegL":"mL","hindlegR":"hR","hindlegL":"hL"}.get(k, k[:6])
         for k in KP_NAMES]

# ── HELPERS ───────────────────────────────────────────────────────────────────
def find_col(df, *required, exclude=()):
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in required) and not any(k in cl for k in exclude):
            return c
    return None

def load_npz(path):
    if not Path(path).exists():
        return {}
    m = np.load(path, allow_pickle=True)
    d = m["metrics"].item() if "metrics" in m else dict(m)
    out = {}
    # mOKS — stored as {"mOKS": value}
    try:
        moks = d["mOKS"]
        out["mOKS"] = float(moks["mOKS"]) if isinstance(moks, dict) else float(np.squeeze(moks))
    except: pass
    # mAP / mAR — keys are "oks_voc.mAP" / "oks_voc.mAR"
    try:
        vm = d["voc_metrics"]
        out["mAP"] = float(vm.get("oks_voc.mAP", vm.get("mAP", float("nan"))))
        out["mAR"] = float(vm.get("oks_voc.mAR", vm.get("mAR", float("nan"))))
    except: pass
    # distance metrics
    try:
        dm = d["distance_metrics"]
        for k in ["avg","p50","p90","p95","p99"]:
            if k in dm: out[f"dist_{k}"] = float(np.nanmean(dm[k]))
    except: pass
    # PCK metrics
    try:
        pm = d["pck_metrics"]
        out["mPCK"]    = float(pm["mPCK"])
        out["PCK@5px"] = float(pm.get("PCK@5", pm.get("pck@5", float("nan"))))
    except: pass
    # visibility metrics
    try:
        vm = d["visibility_metrics"]
        out.update({"vis_precision": float(vm["precision"]),
                    "vis_recall"   : float(vm["recall"]),
                    "vis_FP": int(vm["fp"]), "vis_FN": int(vm["fn"])})
    except: pass
    return out

def load_slp_pts(path):
    if not Path(path).exists():
        return None
    labels = sio.load_slp(str(path))
    result = {}
    for lf in labels:
        if not lf.instances: continue
        inst = lf.instances[0]
        try:
            pts = inst.numpy()
        except Exception:
            try:
                pts = np.array([[p.x, p.y] for p in inst.points], dtype=float)
            except:
                continue
        result[lf.frame_idx] = pts.astype(float)
    return result

def compute_custom(pts_pr, pts_gt, tau=5.0):
    distances = [[] for _ in range(N_KP)]
    tp = np.zeros(N_KP, int); fp = np.zeros(N_KP, int)
    tn = np.zeros(N_KP, int); fn = np.zeros(N_KP, int)
    n_vis = np.zeros(N_KP, int); n_invis = np.zeros(N_KP, int)
    frames = sorted(pts_gt.keys())
    frame_error = {}
    for fidx in frames:
        if fidx not in pts_pr: continue
        gt = pts_gt[fidx]; pr = pts_pr[fidx]
        ferr = False
        for k in range(N_KP):
            gv = not (np.isnan(gt[k,0]) or np.isnan(gt[k,1]))
            pv = not (np.isnan(pr[k,0]) or np.isnan(pr[k,1]))
            if gv:
                n_vis[k] += 1
                if pv:
                    tp[k] += 1
                    d = float(np.hypot(pr[k,0]-gt[k,0], pr[k,1]-gt[k,1]))
                    distances[k].append(d)
                    if d > tau: ferr = True
                else:
                    fn[k] += 1; ferr = True
            else:
                n_invis[k] += 1
                if pv:   fp[k] += 1; ferr = True
                else:    tn[k] += 1
        frame_error[fidx] = ferr
    mpe     = np.array([np.nanmean(distances[k]) if distances[k] else np.nan for k in range(N_KP)])
    fp_rate = np.where(n_invis > 0, fp / n_invis, np.nan)
    fn_rate = np.where(n_vis   > 0, fn / n_vis,   np.nan)
    td = np.full(N_KP, np.nan)
    for k in range(N_KP):
        prev = None; disps = []
        for fidx in frames:
            if fidx not in pts_pr: continue
            pr = pts_pr[fidx]
            pv = not (np.isnan(pr[k,0]) or np.isnan(pr[k,1]))
            if pv:
                if prev is not None: disps.append(float(np.hypot(pr[k,0]-prev[0], pr[k,1]-prev[1])))
                prev = pr[k]
            else: prev = None
        if disps: td[k] = np.mean(disps)
    err_counts = fp + fn + np.array([sum(1 for d in distances[k] if d > tau) for k in range(N_KP)])
    total_kpfr = n_vis + n_invis
    cr_kp      = np.where(total_kpfr > 0, err_counts / total_kpfr * 100, np.nan)
    cr_frame   = sum(frame_error.values()) / len(frame_error) * 100 if frame_error else np.nan
    return dict(mpe=mpe, fp_rate=fp_rate, fn_rate=fn_rate,
                tp=tp, fp=fp, tn=tn, fn=fn, td=td,
                cr_kp=cr_kp, cr_frame=cr_frame,
                global_fp=fp.sum()/n_invis.sum() if n_invis.sum()>0 else np.nan,
                global_fn=fn.sum()/n_vis.sum()   if n_vis.sum()>0   else np.nan,
                n_vis=n_vis, n_invis=n_invis, n_frames=len(frame_error))

def eval_model(model_dir, test_gt_path=None):
    log_path = Path(model_dir) / "training_log.csv"
    log_df = None
    if log_path.exists():
        log_df = pd.read_csv(log_path)
        log_df.columns = [c.replace("/","_") for c in log_df.columns]
    sleap_m, custom_m = {}, {}
    for split in ("train", "val", "test"):
        sleap_m[split]  = load_npz(Path(model_dir) / f"metrics.{split}.0.npz")
        pr_path = Path(model_dir) / f"labels_pr.{split}.0.slp"
        gt_path = Path(model_dir) / f"labels_gt.{split}.0.slp"
        # TEST fallback: use original labels file as GT if labels_gt.test.0.slp absent
        if split == "test" and not gt_path.exists() and test_gt_path:
            gt_path = Path(test_gt_path)
        pts_pr = load_slp_pts(pr_path)
        pts_gt = load_slp_pts(gt_path)
        if pts_pr and pts_gt:
            custom_m[split] = compute_custom(pts_pr, pts_gt, TAU_PX)
        else:
            custom_m[split] = None
    return sleap_m, custom_m, log_df

def show_saved(path):
    display(Image(str(path)))

# ── CHECK MODEL EXISTS ────────────────────────────────────────────────────────
if not (NEW_MODEL_DIR / "best.ckpt").exists():
    if MODE == "colab":
        print("Colab model not yet downloaded.")
        print(f"Download from Drive: MyDrive/sleap/models/{RUN_NAME}/")
        print(f"Place in: {MODELS_DIR}")
    else:
        print(f"Model not found: {NEW_MODEL_DIR}")
        print("Check Step 4 output for errors.")
    raise SystemExit

print(f"Fine-tuned model : {RUN_NAME}")
print(f"Base model       : {BASE_MODEL_NAME}")

# ── 1. LOSS CURVES ────────────────────────────────────────────────────────────
sm, cm, log_df = eval_model(NEW_MODEL_DIR, test_gt_path=ORIG_TEST_SLP)

if log_df is not None:
    ep_col    = find_col(log_df, "epoch")
    train_col = find_col(log_df, "train", "loss", exclude=("confmaps",))
    val_col   = find_col(log_df, "val",   "loss", exclude=("confmaps",))
    lr_col    = find_col(log_df, "learning_rate") or find_col(log_df, "lr")
    val_data  = log_df[[ep_col, val_col]].dropna() if val_col else None

    ncols = 2 if lr_col else 1
    fig, axes = plt.subplots(1, ncols, figsize=(7*ncols, 4))
    if ncols == 1: axes = [axes]
    ax = axes[0]
    if train_col:
        t = log_df[[ep_col, train_col]].dropna()
        if not t.empty: ax.plot(t[ep_col], t[train_col], label="Train", color="steelblue", lw=2)
    if val_data is not None and not val_data.empty:
        ax.plot(val_data[ep_col], val_data[val_col], label="Val", color="tomato", lw=2, marker="o")
    ax.set(xlabel="Epoch", ylabel="Loss", title=f"Loss Curves — {RUN_NAME}")
    ax.legend(); ax.grid(True, alpha=0.3)
    if lr_col:
        t2 = log_df[[ep_col, lr_col]].dropna()
        axes[1].semilogy(t2[ep_col], t2[lr_col], color="mediumseagreen", lw=2)
        axes[1].set(xlabel="Epoch", ylabel="LR (log)", title="LR Schedule")
        axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    out = NEW_MODEL_DIR / "training_curves.png"
    plt.savefig(str(out), dpi=150); plt.close()
    show_saved(out)

    if val_data is not None and not val_data.empty:
        best_val   = val_data[val_col].min()
        best_epoch = int(val_data.loc[val_data[val_col].idxmin(), ep_col])
        print(f"\nEpochs run : {int(val_data[ep_col].max())+1}")
        print(f"Best val   : {best_val:.6f}  (epoch {best_epoch})")

# ── 2. SLEAP BUILT-IN METRICS ─────────────────────────────────────────────────
print(f"\n{'='*65}")
print("SLEAP built-in metrics")
print(f"{'='*65}")
for split in ("train", "val", "test"):
    ms = sm.get(split, {})
    if ms:
        print(f"\n  {split.upper()}")
        for k, v in ms.items():
            print(f"    {k:<22} {v:.4f}" if isinstance(v, float) else f"    {k:<22} {v}")
    else:
        print(f"\n  {split.upper()} : metrics.npz not found")

# ── 3. CUSTOM METRICS ─────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("Custom metrics")
print(f"{'='*65}")
for split in ("train", "val", "test"):
    res = cm.get(split)
    if res is None:
        print(f"\n  {split.upper()} : labels_pr/gt .slp not found")
        continue
    print(f"\n  {split.upper()}")
    print(f"    MPE (mean)      : {np.nanmean(res['mpe']):.3f} px")
    print(f"    FP rate (global): {res['global_fp']*100:.2f}%")
    print(f"    FN rate (global): {res['global_fn']*100:.2f}%")
    print(f"    Tracking Drift  : {np.nanmean(res['td']):.3f} px/frame")
    print(f"    CR_frame        : {res['cr_frame']:.1f}%")
    print(f"    CR_kp (mean)    : {np.nanmean(res['cr_kp']):.1f}%")

# ── 4. PER-KEYPOINT PLOTS ─────────────────────────────────────────────────────
for split, res in cm.items():
    if res is None: continue
    C = "steelblue" if split == "train" else ("tomato" if split == "val" else "mediumseagreen")
    fig, axes = plt.subplots(2, 3, figsize=(17, 8))
    fig.suptitle(f"{RUN_NAME} — {split.upper()} per-keypoint", fontsize=11)
    ax = axes[0,0]; b = ax.bar(SHORT, res["mpe"], color=C, alpha=0.85)
    ax.bar_label(b, fmt="%.2f", fontsize=7, padding=2)
    ax.axhline(TAU_PX, color="red", ls="--", lw=1, alpha=0.6, label=f"tau={TAU_PX}px")
    ax.set(ylabel="px", title="MPE"); ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[0,1]; b = ax.bar(SHORT, res["fp_rate"]*100, color="salmon", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title="False Contact Rate"); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[0,2]; b = ax.bar(SHORT, res["fn_rate"]*100, color="gold", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title="Missed Contact Rate"); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[1,0]; b = ax.bar(SHORT, res["td"], color="mediumseagreen", alpha=0.85)
    ax.bar_label(b, fmt="%.2f", fontsize=7, padding=2)
    ax.set(ylabel="px/frame", title="Tracking Drift"); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[1,1]; b = ax.bar(SHORT, res["cr_kp"], color="mediumpurple", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title=f"CR_kp (tau={TAU_PX}px)"); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[1,2]; x = np.arange(N_KP); w = 0.20
    ax.bar(x-1.5*w, res["tp"], w, label="TP", color="limegreen", alpha=0.8)
    ax.bar(x-0.5*w, res["fp"], w, label="FP", color="red",       alpha=0.7)
    ax.bar(x+0.5*w, res["fn"], w, label="FN", color="orange",    alpha=0.8)
    ax.bar(x+1.5*w, res["tn"], w, label="TN", color="silver",    alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(SHORT, rotation=30, fontsize=8)
    ax.set(ylabel="count", title="Detection Counts"); ax.legend(fontsize=8)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    out = NEW_MODEL_DIR / f"eval_kp_metrics_{split}.png"
    plt.savefig(str(out), dpi=150); plt.close()
    show_saved(out)

# ── 5. COMPARISON vs BASE MODEL ──────────────────────────────────────────────
from IPython.display import display as _ipy_display

print()
print('='*65)
print(f"COMPARISON: fine-tuned vs base model (VAL set)")
print('='*65)

base_sm, base_cm, _ = eval_model(BASE_MODEL_DIR, test_gt_path=ORIG_TEST_SLP)

def fmt(v, d=3):
    try:
        fv = float(v)
        if fv != fv: return "—"  # nan check
        return f"{fv:.{d}f}"
    except: return "—"

def compute_delta(new_val, old_val, lower_is_better=True):
    """Returns (formatted_delta_str, is_better).  is_better: True/False/None."""
    try:
        nv, ov = float(new_val), float(old_val)
        if nv != nv or ov != ov: return "—", None  # nan check
        d = nv - ov
        sign = "+" if d > 0 else ""
        better = (d < 0) == lower_is_better
        return f"{sign}{d:.4f}", better
    except:
        return "—", None

rows, win_flags = [], []

for key, label, lib in [
    ("dist_avg",      "Avg dist (px)",   True),
    ("dist_p50",      "p50 dist (px)",   True),
    ("dist_p90",      "p90 dist (px)",   True),
    ("mOKS",          "mOKS",            False),
    ("mAP",           "mAP",             False),
    ("mAR",           "mAR",             False),
    ("mPCK",          "mPCK",            False),
    ("PCK@5px",       "PCK@5px",         False),
    ("vis_precision", "Vis Precision",   False),
    ("vis_recall",    "Vis Recall",      False),
]:
    bv = base_sm.get("val", {}).get(key, float("nan"))
    nv = sm.get("val", {}).get(key, float("nan"))
    delta, is_better = compute_delta(nv, bv, lib)
    rows.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
    win_flags.append(is_better)

bcm = base_cm.get("val"); ncm = cm.get("val")
for attr, label, lib in [
    ("global_fp", "FP rate (%)",  True),
    ("global_fn", "FN rate (%)",  True),
    ("cr_frame",  "CR_frame (%)", True),
]:
    bv = (bcm[attr]*100 if attr in ("global_fp","global_fn") else bcm[attr]) if bcm else float("nan")
    nv = (ncm[attr]*100 if attr in ("global_fp","global_fn") else ncm[attr]) if ncm else float("nan")
    delta, is_better = compute_delta(nv, bv, lib)
    rows.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
    win_flags.append(is_better)

for attr, label in [("mpe", "MPE px"), ("td", "TD px/fr"), ("cr_kp", "CR_kp %")]:
    bv = np.nanmean(bcm[attr]) if bcm else float("nan")
    nv = np.nanmean(ncm[attr]) if ncm else float("nan")
    delta, is_better = compute_delta(nv, bv, lower_is_better=True)
    rows.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
    win_flags.append(is_better)

df_cmp = pd.DataFrame(rows)
df_cmp["Result"] = [
    "✓ Better" if f is True else ("✗ Worse" if f is False else "  —")
    for f in win_flags
]

# ── Styled HTML table ──────────────────────────────────────────────────────────────────────
def _style_result(val):
    if "✓" in str(val):
        return ("background-color:#c6efce; color:#276221; font-weight:bold; "
                "text-align:center; border-radius:3px")
    if "✗" in str(val):
        return ("background-color:#ffc7ce; color:#9c0006; font-weight:bold; "
                "text-align:center; border-radius:3px")
    return "color:#888; text-align:center"

def _style_rows(row):
    ib = win_flags[row.name]
    out = {c: "" for c in row.index}
    if ib is True:
        out["Fine-tuned"] = "background-color:#eafaea; color:#276221; font-weight:bold"
        out["Delta"]      = "color:#276221; font-weight:bold"
    elif ib is False:
        out["Fine-tuned"] = "background-color:#fff0f0; color:#9c0006; font-weight:bold"
        out["Delta"]      = "color:#9c0006; font-weight:bold"
    return pd.Series(out)

n_better = sum(1 for f in win_flags if f is True)
n_worse  = sum(1 for f in win_flags if f is False)

styled = (
    df_cmp.style
    .map(_style_result, subset=["Result"])
    .apply(_style_rows, axis=1)
    .set_table_styles([
        {"selector": "th",
         "props": [("background-color", "#e8e8e8"), ("font-weight", "bold"),
                   ("text-align", "center"), ("padding", "6px 12px")]},
        {"selector": "th:first-child",
         "props": [("text-align", "left")]},
        {"selector": "td",
         "props": [("padding", "5px 12px"), ("text-align", "center"),
                   ("border-bottom", "1px solid #e0e0e0")]},
        {"selector": "td:first-child",
         "props": [("text-align", "left"), ("font-weight", "bold")]},
        {"selector": "caption",
         "props": [("font-size", "1.05em"), ("font-weight", "bold"),
                   ("margin-bottom", "8px"), ("caption-side", "top")]},
    ])
    .hide(axis="index")
    .set_caption(
        f"Fine-tuned vs Base (VAL set) — "
        f"{n_better}/{len(win_flags)} metrics improved, "
        f"{n_worse}/{len(win_flags)} regressed"
    )
)
_ipy_display(styled)

df_cmp.to_csv(NEW_MODEL_DIR / "comparison_vs_base.csv", index=False)
print(f"Saved: comparison_vs_base.csv  ({n_better}/{len(win_flags)} better, {n_worse}/{len(win_flags)} worse)")


# ── 5b. COMPARISON vs BASE MODEL (TEST set) ───────────────────────────────────
_has_test = (cm.get("test") is not None) or (base_cm.get("test") is not None)
if not _has_test:
    print("\nTEST comparison: no TEST metrics found for either model.")
    print("Run Step 4.5 to generate them, or re-run Steps 3+4.")
else:
    print()
    print('='*65)
    print("COMPARISON: fine-tuned vs base model (TEST set)")
    print('='*65)
    print()

    rows_t, win_flags_t = [], []

    for key, label, lib in [
        ("dist_avg",      "Avg dist (px)",   True),
        ("dist_p50",      "p50 dist (px)",   True),
        ("dist_p90",      "p90 dist (px)",   True),
        ("mOKS",          "mOKS",            False),
        ("mAP",           "mAP",             False),
        ("mAR",           "mAR",             False),
        ("mPCK",          "mPCK",            False),
        ("PCK@5px",       "PCK@5px",         False),
        ("vis_precision", "Vis Precision",   False),
        ("vis_recall",    "Vis Recall",      False),
    ]:
        bv = base_sm.get("test", {}).get(key, float("nan"))
        nv = sm.get("test", {}).get(key, float("nan"))
        delta, is_better = compute_delta(nv, bv, lib)
        rows_t.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
        win_flags_t.append(is_better)

    bcm_t = base_cm.get("test"); ncm_t = cm.get("test")
    for attr, label, lib in [
        ("global_fp", "FP rate (%)",  True),
        ("global_fn", "FN rate (%)",  True),
        ("cr_frame",  "CR_frame (%)", True),
    ]:
        bv = (bcm_t[attr]*100 if attr in ("global_fp","global_fn") else bcm_t[attr]) if bcm_t else float("nan")
        nv = (ncm_t[attr]*100 if attr in ("global_fp","global_fn") else ncm_t[attr]) if ncm_t else float("nan")
        delta, is_better = compute_delta(nv, bv, lib)
        rows_t.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
        win_flags_t.append(is_better)

    for attr, label in [("mpe", "MPE px"), ("td", "TD px/fr"), ("cr_kp", "CR_kp %")]:
        bv = np.nanmean(bcm_t[attr]) if bcm_t else float("nan")
        nv = np.nanmean(ncm_t[attr]) if ncm_t else float("nan")
        delta, is_better = compute_delta(nv, bv, lower_is_better=True)
        rows_t.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
        win_flags_t.append(is_better)

    df_cmp_t = pd.DataFrame(rows_t)
    df_cmp_t["Result"] = [
        "\u2713 Better" if f is True else ("\u2717 Worse" if f is False else "  \u2014")
        for f in win_flags_t
    ]

    def _style_rows_t(row):
        ib = win_flags_t[row.name]
        out = {c: "" for c in row.index}
        if ib is True:
            out["Fine-tuned"] = "background-color:#eafaea; color:#276221; font-weight:bold"
            out["Delta"]      = "color:#276221; font-weight:bold"
        elif ib is False:
            out["Fine-tuned"] = "background-color:#fff0f0; color:#9c0006; font-weight:bold"
            out["Delta"]      = "color:#9c0006; font-weight:bold"
        return pd.Series(out)

    n_better_t = sum(1 for f in win_flags_t if f is True)
    n_worse_t  = sum(1 for f in win_flags_t if f is False)

    styled_t = (
        df_cmp_t.style
        .map(_style_result, subset=["Result"])
        .apply(_style_rows_t, axis=1)
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#e8e8e8"), ("font-weight", "bold"),
                       ("text-align", "center"), ("padding", "6px 12px")]},
            {"selector": "th:first-child",
             "props": [("text-align", "left")]},
            {"selector": "td",
             "props": [("padding", "5px 12px"), ("text-align", "center"),
                       ("border-bottom", "1px solid #e0e0e0")]},
            {"selector": "td:first-child",
             "props": [("text-align", "left"), ("font-weight", "bold")]},
            {"selector": "caption",
             "props": [("font-size", "1.05em"), ("font-weight", "bold"),
                       ("margin-bottom", "8px"), ("caption-side", "top")]},
        ])
        .hide(axis="index")
        .set_caption(
            f"Fine-tuned vs Base (TEST set) \u2014 "
            f"{n_better_t}/{len(win_flags_t)} metrics improved, "
            f"{n_worse_t}/{len(win_flags_t)} regressed"
        )
    )
    _ipy_display(styled_t)

    df_cmp_t.to_csv(NEW_MODEL_DIR / "comparison_vs_base_test.csv", index=False)
    print(f"Saved: comparison_vs_base_test.csv  ({n_better_t}/{len(win_flags_t)} better, {n_worse_t}/{len(win_flags_t)} worse)")

# ── 7. SUMMARY ────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("To use this model, set in Bulk_Pipeline.ipynb FLAGS cell:")
print(f'  SLEAP_MODEL_NAME = "{RUN_NAME}"')
print(f"{'='*65}")


In [ ]:
# ── Step 5 Quick Recovery ────────────────────────────────────────────────────
# Run ONLY this cell (+ the Step 5 cell below) if the kernel died after training.
# Set the two names below to match your completed run, then run Step 5.

from pathlib import Path

_BASE_DIR   = Path().resolve()
MODELS_DIR  = _BASE_DIR / "Models" / "SLEAP_Models"

# Fill these in — copy from the Step 4 output ("Run : ..." and "Output : ...")
RUN_NAME        = "drosophila_unet64_ft_976frames_260415_094826"   # ← your completed run name
BASE_MODEL_NAME = "drosophila_unet64_setB_260407_092817"

NEW_MODEL_DIR  = MODELS_DIR / RUN_NAME
BASE_MODEL_DIR = MODELS_DIR / BASE_MODEL_NAME
MODE           = "local"

# Also restore TEST path (used in Step 4.5 and Step 5)
ORIG_TEST_SLP  = _BASE_DIR / "SLEAP_Dataset" / "Drosophila_TEST_set_setB.slp"

if (NEW_MODEL_DIR / "best.ckpt").exists():
    print(f"Model found : {RUN_NAME}")
    print(f"Proceed to Step 5.")
else:
    print(f"ERROR: best.ckpt not found at {NEW_MODEL_DIR}")
